In [1]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client import models as qdrant_models
import json
from pathlib import Path

In [2]:
load_dotenv()

# cloud_inference=True requis pour générer le BM25 natif via model="Qdrant/bm25"
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    cloud_inference=True,
)

In [3]:
def create_qdrant_hybrid_collection(
    collection_name: str,
    vector_size: int = 1536,
):
    """
    Crée une collection Qdrant adaptée à l'hybride :
    - dense vector nommé "dense" (embeddings)
    - sparse vector nommé "bm25" (BM25 natif Qdrant)

    ⚠️ Si la collection existe déjà, ne fait rien.
    """
    existing_collections = [c.name for c in qdrant_client.get_collections().collections]
    if collection_name in existing_collections:
        print(f"ℹ️ Collection déjà existante : {collection_name}")
        return qdrant_client

    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config={
            # Dense embeddings
            "dense": qdrant_models.VectorParams(
                size=vector_size,
                distance=qdrant_models.Distance.COSINE,
            )
        },
        sparse_vectors_config={
            # Sparse BM25 slot (rempli via qdrant_models.Document(model="Qdrant/bm25"))
            "bm25": qdrant_models.SparseVectorParams()
        },
    )

    print(f"✅ Collection hybride créée : {collection_name}")
    return qdrant_client


In [8]:
COLLECTION_NAME_HYBRIDE = "rag_noob_collection_hybride"
create_qdrant_hybrid_collection(collection_name=COLLECTION_NAME_HYBRIDE, vector_size=1536)

collections = qdrant_client.get_collections().collections
print([c.name for c in collections])

ℹ️ Collection déjà existante : rag_noob_collection_hybride
['rag_noob_collection_hybride', 'rag_noob_collection']


In [11]:
def index_doc_hybrid_to_qdrant(
    doc_name: str,
    batch_size: int = 64,
    collection_name: str = None,
    text_field: str = "content",
):
    if collection_name is None:
        collection_name = COLLECTION_NAME_HYBRIDE

    vectors_path = Path("../data/vectors") / f"{doc_name}.json"
    if not vectors_path.exists():
        raise FileNotFoundError(f"Fichier vecteurs introuvable : {vectors_path}")

    items = json.loads(vectors_path.read_text(encoding="utf-8"))
    total = len(items)

    print(f"📦 {doc_name} → {total} points hybrides à indexer dans '{collection_name}'")

    points_buffer = []
    upserted = 0

    for item in items:
        payload = item.get("payload", {}) or {}
        text = payload.get(text_field, "")

        points_buffer.append(
            qdrant_models.PointStruct(
                id=item["id"],
                vector={
                    "dense": item["vector"],
                    "bm25": qdrant_models.Document(
                        text=text,
                        model="Qdrant/bm25",
                    ),
                },
                payload=payload,
            )
        )

        if len(points_buffer) >= batch_size:
            qdrant_client.upsert(collection_name=collection_name, points=points_buffer)
            upserted += len(points_buffer)
            points_buffer = []
            print(f"✅ upsert {upserted}/{total}")

    if points_buffer:
        qdrant_client.upsert(collection_name=collection_name, points=points_buffer)
        upserted += len(points_buffer)

    print(f"🎉 Terminé : {upserted}/{total} points hybrides indexés")
    return upserted


In [12]:
def index_all_docs_hybride_to_qdrant():
    """
    Indexe tous les fichiers ../data/vectors/{doc_name}.json dans Qdrant
    en réutilisant index_doc_vectors_to_qdrant().
    """
    vectors_dir = Path("../data/vectors")
    if not vectors_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable : {vectors_dir}")

    vector_files = sorted(vectors_dir.glob("*.json"))
    if not vector_files:
        print("⚠️ Aucun fichier .json trouvé dans ../data/vectors/")
        return {}

    results = {}
    total_docs = len(vector_files)

    print(f"🚀 Indexation Qdrant pour {total_docs} documents")

    for i, vf in enumerate(vector_files, start=1):
        doc_name = vf.stem
        print(f"\n📄 [{i}/{total_docs}] Indexation : {doc_name}")

        try:
            n = index_doc_hybrid_to_qdrant(doc_name=doc_name)
            results[doc_name] = {"status": "ok", "n_vectors": n}
        except Exception as e:
            print(f"❌ Erreur pour {doc_name} : {e}")
            results[doc_name] = {"status": "error", "error": str(e)}

    print("\n🎉 Indexation batch terminée")
    return results

In [13]:
index_all_docs_hybride_to_qdrant()

🚀 Indexation Qdrant pour 7 documents

📄 [1/7] Indexation : GDO-Interventions-en-milieu-agricole-2019-V2
📦 GDO-Interventions-en-milieu-agricole-2019-V2 → 55 points hybrides à indexer dans 'rag_noob_collection_hybride'
🎉 Terminé : 55/55 points hybrides indexés

📄 [2/7] Indexation : GDO-interventions-silos-VF-09-2019
📦 GDO-interventions-silos-VF-09-2019 → 41 points hybrides à indexer dans 'rag_noob_collection_hybride'
🎉 Terminé : 41/41 points hybrides indexés

📄 [3/7] Indexation : GDO-Operations-Presence-Electricite
📦 GDO-Operations-Presence-Electricite → 92 points hybrides à indexer dans 'rag_noob_collection_hybride'
✅ upsert 64/92
🎉 Terminé : 92/92 points hybrides indexés

📄 [4/7] Indexation : GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC
📦 GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC → 34 points hybrides à indexer dans 'rag_noob_collection_hybride'
🎉 Terminé : 34/34 points hybrides indexés

📄 [5/7] Indexation : GDO_ExerciceCdtConduiteOperations_BDFE_D

{'GDO-Interventions-en-milieu-agricole-2019-V2': {'status': 'ok',
  'n_vectors': 55},
 'GDO-interventions-silos-VF-09-2019': {'status': 'ok', 'n_vectors': 41},
 'GDO-Operations-Presence-Electricite': {'status': 'ok', 'n_vectors': 92},
 'GDO_2eme-edition-Prevention-risque-toxicite-fumees_BDFE_DGSCGC': {'status': 'ok',
  'n_vectors': 34},
 'GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2': {'status': 'ok',
  'n_vectors': 77},
 'GDO_interventions_a_bord_bateaux_en_eaux_interieures': {'status': 'ok',
  'n_vectors': 46},
 'GDO_Interventions_dans_les_eoliennes_2019': {'status': 'ok',
  'n_vectors': 18}}

RETRIEVING HYBRIDE

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from base_function import embed_content, search_top_chunks_bm25, search_top_chunks_semantique

In [ ]:
import os
from qdrant_client import QdrantClient
from qdrant_client import models as qdrant_models

def search_top_chunks_hybrid(
    question: str,
    top_k: int = 5,
    query_filter=None,
):
    qdrant_client = QdrantClient(
        url=os.getenv("QDRANT_URL"),
        api_key=os.getenv("QDRANT_API_KEY"),
        cloud_inference=True,  # nécessaire si tu utilises Document(model="Qdrant/bm25")
    )

    question_vector = embed_content({"content": question})

    res = qdrant_client.query_points(
        collection_name=COLLECTION_NAME_HYBRIDE,
        prefetch=[
            # 1) Dense
            qdrant_models.Prefetch(
                query=question_vector,
                using="dense",
                limit=top_k,
                filter=query_filter,
            ),
            # 2) BM25 natif
            qdrant_models.Prefetch(
                query=qdrant_models.Document(
                    text=question,
                    model="Qdrant/bm25",
                ),
                using="bm25",
                limit=top_k,
                filter=query_filter,
            ),
        ],
        # fusion côté serveur 
        query=qdrant_models.FusionQuery(
            fusion=qdrant_models.Fusion.RRF
        ),
        limit=top_k,
        with_payload=True,
    )

    chunks = []
    for rank, hit in enumerate(res.points, start=1):
        payload = hit.payload or {}
        chunks.append({
            "rank": rank,
            "score": getattr(hit, "score", None),
            "doc_name": payload.get("doc_name"),
            "chunk_id": payload.get("chunk_id"),
            "content": payload.get("content"),
        })
    return chunks


In [7]:
question = "périmètre de sécurité autour d’une éolienne en feu ?"

print("\n--- RAG Hybride ---")
chunks = search_top_chunks_hybrid(question, top_k = 5)

for c in chunks:
    print(f"\n--- Rank {c['rank']} | score={c['score']} ---")
    print(f"Doc: {c['doc_name']} | chunk: {c['chunk_id']}")


--- RAG Hybride ---

--- Rank 1 | score=1.0 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 76

--- Rank 2 | score=0.53333336 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 77

--- Rank 3 | score=0.5 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 75

--- Rank 4 | score=0.25 ---
Doc: GDO_Interventions_dans_les_eoliennes_2019 | chunk: 13

--- Rank 5 | score=0.25 ---
Doc: GDO-Operations-Presence-Electricite | chunk: 73


Comparaison des modes de retrieval

In [9]:
from tabulate import tabulate

def compare_retrieval_strategies(
    question: str,
    top_k: int = 5,
):
    bm25 = search_top_chunks_bm25(question, top_k=top_k)
    semantic = search_top_chunks_semantique(question, top_k=top_k)
    hybrid = search_top_chunks_hybrid(question, top_k=top_k)

    def key(c):
        return (c["doc_name"], c["chunk_id"])

    bm25_rank = {key(c): c["rank"] for c in bm25}
    semantic_rank = {key(c): c["rank"] for c in semantic}
    hybrid_rank = {key(c): c["rank"] for c in hybrid}

    rows = []
    for c in hybrid:
        k = key(c)
        rows.append([
            f"{c['doc_name']} | chunk {c['chunk_id']}",
            bm25_rank.get(k, ""),
            semantic_rank.get(k, ""),
            hybrid_rank.get(k, ""),
        ])

    rows.sort(key=lambda r: r[3])  # tri par rank hybride

    print("\nQUESTION :")
    print(question)
    print()

    print(
        tabulate(
            rows,
            headers=["Chunk", "Rank BM25", "Rank Sémantique", "Rank Hybride"],
            tablefmt="github",  # ou "fancy_grid", "grid"
        )
    )


In [10]:
compare_retrieval_strategies(
    "Intervention sur éolienne, quelles infos sont indispensables pour envoyer les bons secours ?"
)


QUESTION :
Intervention sur éolienne, quelles infos sont indispensables pour envoyer les bons secours ?

| Chunk                                                       | Rank BM25   | Rank Sémantique   |   Rank Hybride |
|-------------------------------------------------------------|-------------|-------------------|----------------|
| GDO_Interventions_dans_les_eoliennes_2019 | chunk 13        |             | 1                 |              1 |
| GDO_ExerciceCdtConduiteOperations_BDFE_DGSCGC_V2 | chunk 31 | 1           |                   |              2 |
| GDO-Operations-Presence-Electricite | chunk 77              |             | 2                 |              3 |
| GDO_Interventions_dans_les_eoliennes_2019 | chunk 0         |             |                   |              4 |
| GDO-Operations-Presence-Electricite | chunk 76              |             | 3                 |              5 |


In [3]:
from base_function import ask_question_with_hybrid_rag

In [5]:
question = "Intervention sur éolienne, quelles infos sont indispensables pour envoyer les bons secours ?"

print("\n--- RAG hybride ---")
model_response, answer, memory = ask_question_with_hybrid_rag(
    question,
    top_k=5
)
print(answer)


--- RAG hybride ---
Pour envoyer les bons secours lors d'une intervention sur une éolienne, les informations indispensables incluent :

1. **Localisation** : Commune, nom du parc, numéro de l’éolienne, et géolocalisation dans un système d’information géographique.
2. **Description de la problématique** : Type de sinistre, nombre de personnes en difficulté, leur pathologie, et leur localisation.
3. **Nature du requérant** : Identité de la personne qui appelle (témoin, agent de maintenance, exploitant, centre de surveillance, etc.) et consignes à lui donner avant l’arrivée des secours.
4. **Facteurs aggravants** : Éléments tels que le nombre d'appels, conditions climatiques, heure, rassemblement à proximité, coupure électrique du site, et fermeture de la porte d’accès.
5. **Caractéristiques des éoliennes** : Hauteur de mât, localisation des arrêts d’urgence, moyens de communication disponibles, et modalités d’accès dans l’éolienne.
6. **Coordonnées du responsable de l’exploitation** : P